In [1]:
import pandas as pd
import numpy as np
import csv
import matplotlib.pyplot as plt
from decimal import *
import re
from scipy.interpolate import interp1d

#### Based on the Manaus_Population_Correction.ipynb notebook, which has already estimated the yearly rural population from IBGE data, we have:

In [3]:
# pop_manaus_2000 = 1403796 ### https://biblioteca.ibge.gov.br/index.php/biblioteca-catalogo?view=detalhes&id=7308 (Tabelas Excel > AM > UF > Tabela 15)
# pop_manaus_2010 = 1802014 ### https://www.ibge.gov.br/estatisticas/sociais/populacao/9662-censo-demografico-2010.html?=&t=resultados (Amazonas > Tabela 2.1.3)
# pop_manaus_2022 = 2063689 ### https://cidades.ibge.gov.br/brasil/am/manaus/panorama, https://www.ibge.gov.br/cidades-e-estados/am/manaus.html
# pop_manaus_2025 = 2303732 ### ESTIMATED, https://cidades.ibge.gov.br/brasil/am/manaus/panorama, https://www.ibge.gov.br/cidades-e-estados/am/manaus.html


# ibge_pop_data = {
#     'YEAR': [2000, 2010, 2022, 2025],
#     'POPULATION': [pop_manaus_2000, pop_manaus_2010, pop_manaus_2022, pop_manaus_2025]
# }

# ibge_pop_data_df = pd.DataFrame(ibge_pop_data)

# ibge_pop_data_df.set_index('YEAR', inplace=True)

# ibge_pop_data_df_interpolated = ibge_pop_data_df.reindex(range(2000, 2026))  # Reindex to include all years from 2000 to 2025
# ibge_pop_data_df_interpolated['POPULATION'] = ibge_pop_data_df_interpolated['POPULATION'].interpolate(method='linear')

# ibge_pop_data_df_interpolated.reset_index(inplace=True)

# ibge_pop_data_df_interpolated['POPULATION'] = ibge_pop_data_df_interpolated['POPULATION'].astype(int)

# ibge_pop_data_df_interpolated['MUNIC_RES'] = '130260'
# ibge_pop_data_df_interpolated

ibge_pop_data_df = pd.read_csv("../data/ibge_manaus_rural_population_data_2000_2025.csv")
ibge_pop_data_df

,year,total_population,rural_proportion,rural_population
0,2000,1403796,0.006450,9053.849372
1,2001,1443617,0.006296,9088.952950
2,2002,1483439,0.006146,9117.256294
3,2003,1523261,0.006000,9139.057233
4,2004,1563083,0.005857,9154.649411
5,2005,1602905,0.005717,9164.316175
6,2006,1642726,0.005581,9168.325316
7,2007,1682548,0.005448,9166.951842
8,2008,1722370,0.005319,9160.444394
9,2009,1762192,0.005192,9149.048271


In [7]:
ibge_rural_pop_data_df_2012_2025 = ibge_pop_data_df[(ibge_pop_data_df.get('year') >= 2012)
                                                         & (ibge_pop_data_df.get('year') <= 2025)].copy()

ibge_rural_pop_data_df_2012_2025['year'] = ibge_rural_pop_data_df_2012_2025['year'].astype(int)

# Create a date column (Jan 1 of each year)
ibge_rural_pop_data_df_2012_2025['date'] = pd.to_datetime(ibge_rural_pop_data_df_2012_2025['year'], format='%Y')

# Set as index
ibge_rural_pop_data_df_2012_2025 = ibge_rural_pop_data_df_2012_2025.set_index('date')

# Create full daily range
daily_index = pd.date_range(start='2012-01-01', end='2025-12-31', freq='D')

# Reindex
ibge_rural_pop_data_df_2012_2025_daily = ibge_rural_pop_data_df_2012_2025.reindex(daily_index)

ibge_rural_pop_data_df_2012_2025_daily['daily_pop'] = ibge_rural_pop_data_df_2012_2025_daily['total_population'].interpolate(method='linear')
ibge_rural_pop_data_df_2012_2025_daily['daily_rural_pop'] = ibge_rural_pop_data_df_2012_2025_daily['rural_population'].interpolate(method='linear')

clean_ibge_rural_pop_data_df_2012_2025 = ibge_rural_pop_data_df_2012_2025_daily.drop(columns=['year', 'total_population', 'rural_proportion', 'rural_population'])

clean_ibge_rural_pop_data_df_2012_2025

,daily_pop,daily_rural_pop
2012-01-01,1.845626e+06,8913.827042
2012-01-02,1.845686e+06,8913.527958
2012-01-03,1.845745e+06,8913.228874
2012-01-04,1.845805e+06,8912.929789
2012-01-05,1.845864e+06,8912.630705
...,...,...
2025-12-27,2.303732e+06,8133.478083
2025-12-28,2.303732e+06,8133.478083
2025-12-29,2.303732e+06,8133.478083
2025-12-30,2.303732e+06,8133.478083


In [9]:
#### Mean daily growth of population from 2012 to 2025
start_date = clean_ibge_rural_pop_data_df_2012_2025.index.min()
end_date = clean_ibge_rural_pop_data_df_2012_2025.index.max()

start_pop = clean_ibge_rural_pop_data_df_2012_2025.loc[start_date, 'daily_pop']
end_pop = clean_ibge_rural_pop_data_df_2012_2025.loc[end_date, 'daily_pop']

mean_daily_growth = (end_pop - start_pop) / (end_date - start_date).days
mean_daily_growth

np.float64(89.59632309798553)

In [23]:
clean_ibge_rural_pop_data_df_2012_2015 = clean_ibge_rural_pop_data_df_2012_2025.loc['2012-01-01':'2015-12-31']
clean_ibge_rural_pop_data_df_2012_2015

,daily_pop,daily_rural_pop
2012-01-01,1.845626e+06,8913.827042
2012-01-02,1.845686e+06,8913.527958
2012-01-03,1.845745e+06,8913.228874
2012-01-04,1.845805e+06,8912.929789
2012-01-05,1.845864e+06,8912.630705
...,...,...
2015-12-27,1.932552e+06,8478.629576
2015-12-28,1.932612e+06,8478.331415
2015-12-29,1.932672e+06,8478.033255
2015-12-30,1.932732e+06,8477.735095


In [11]:
clean_ibge_rural_pop_data_df_2016_2024 = clean_ibge_rural_pop_data_df_2012_2025.loc['2016-01-01':'2024-12-31']
clean_ibge_rural_pop_data_df_2016_2024

,daily_pop,daily_rural_pop
2016-01-01,1.932851e+06,8477.138775
2016-01-02,1.932911e+06,8476.842288
2016-01-03,1.932970e+06,8476.545800
2016-01-04,1.933030e+06,8476.249313
2016-01-05,1.933089e+06,8475.952826
...,...,...
2024-12-27,2.302639e+06,8132.235261
2024-12-28,2.302858e+06,8132.483826
2024-12-29,2.303076e+06,8132.732390
2024-12-30,2.303295e+06,8132.980955


In [13]:
#### Mean daily growth of population from 2016 to 2024
start_date = clean_ibge_rural_pop_data_df_2016_2024.index.min()
end_date = clean_ibge_rural_pop_data_df_2016_2024.index.max()

start_pop = clean_ibge_rural_pop_data_df_2016_2024.loc[start_date, 'daily_pop']
end_pop = clean_ibge_rural_pop_data_df_2016_2024.loc[end_date, 'daily_pop']

mean_daily_growth = (end_pop - start_pop) / (end_date - start_date).days
mean_daily_growth

np.float64(112.76616360858559)

In [15]:
per_capita_growth_2012_2025 = (
    clean_ibge_rural_pop_data_df_2012_2025['daily_pop']
    .diff()
    .div(clean_ibge_rural_pop_data_df_2012_2025['daily_pop'])
    .mean()
)

print(f"{per_capita_growth_2012_2025:.10f}")

0.0000433610


In [17]:
per_capita_growth_2016_2024 = (
    clean_ibge_rural_pop_data_df_2016_2024['daily_pop']
    .diff()
    .div(clean_ibge_rural_pop_data_df_2016_2024['daily_pop'])
    .mean()
)

print(f"{per_capita_growth_2016_2024:.10f}")

0.0000533717


In [19]:
#### Mean daily growth of rural population from 2012 to 2025
start_date = clean_ibge_rural_pop_data_df_2012_2025.index.min()
end_date = clean_ibge_rural_pop_data_df_2012_2025.index.max()

start_rural_pop = clean_ibge_rural_pop_data_df_2012_2025.loc[start_date, 'daily_rural_pop']
end_rural_pop = clean_ibge_rural_pop_data_df_2012_2025.loc[end_date, 'daily_rural_pop']

mean_daily_growth = (end_rural_pop - start_rural_pop) / (end_date - start_date).days
mean_daily_growth

np.float64(-0.15262056688419526)

In [21]:
#### Mean daily growth of rural population from 2016 to 2024
start_date = clean_ibge_rural_pop_data_df_2016_2024.index.min()
end_date = clean_ibge_rural_pop_data_df_2016_2024.index.max()

start_rural_pop = clean_ibge_rural_pop_data_df_2016_2024.loc[start_date, 'daily_rural_pop']
end_rural_pop = clean_ibge_rural_pop_data_df_2016_2024.loc[end_date, 'daily_rural_pop']

mean_daily_growth = (end_rural_pop - start_rural_pop) / (end_date - start_date).days
mean_daily_growth

np.float64(-0.10462709336151256)

In [25]:
per_capita_rural_growth_2016_2024 = (
    clean_ibge_rural_pop_data_df_2016_2024['daily_rural_pop']
    .diff()
    .div(clean_ibge_rural_pop_data_df_2016_2024['daily_rural_pop'])
    .mean()
)

print(f"{per_capita_rural_growth_2016_2024:.10f}")

-0.0000126002


In [27]:
reset_clean_ibge_rural_pop_data_df_2012_2015 = clean_ibge_rural_pop_data_df_2012_2015.reset_index().copy()
reset_clean_ibge_rural_pop_data_df_2012_2015

,index,daily_pop,daily_rural_pop
0,2012-01-01,1.845626e+06,8913.827042
1,2012-01-02,1.845686e+06,8913.527958
2,2012-01-03,1.845745e+06,8913.228874
3,2012-01-04,1.845805e+06,8912.929789
4,2012-01-05,1.845864e+06,8912.630705
...,...,...,...
1456,2015-12-27,1.932552e+06,8478.629576
1457,2015-12-28,1.932612e+06,8478.331415
1458,2015-12-29,1.932672e+06,8478.033255
1459,2015-12-30,1.932732e+06,8477.735095


In [29]:
reset_clean_ibge_rural_pop_data_df_2016_2024 = clean_ibge_rural_pop_data_df_2016_2024.reset_index().copy()
reset_clean_ibge_rural_pop_data_df_2016_2024

,index,daily_pop,daily_rural_pop
0,2016-01-01,1.932851e+06,8477.138775
1,2016-01-02,1.932911e+06,8476.842288
2,2016-01-03,1.932970e+06,8476.545800
3,2016-01-04,1.933030e+06,8476.249313
4,2016-01-05,1.933089e+06,8475.952826
...,...,...,...
3283,2024-12-27,2.302639e+06,8132.235261
3284,2024-12-28,2.302858e+06,8132.483826
3285,2024-12-29,2.303076e+06,8132.732390
3286,2024-12-30,2.303295e+06,8132.980955


In [31]:
reset_clean_ibge_rural_pop_data_df_2012_2025 = clean_ibge_rural_pop_data_df_2012_2025.reset_index().copy()
reset_clean_ibge_rural_pop_data_df_2012_2025

,index,daily_pop,daily_rural_pop
0,2012-01-01,1.845626e+06,8913.827042
1,2012-01-02,1.845686e+06,8913.527958
2,2012-01-03,1.845745e+06,8913.228874
3,2012-01-04,1.845805e+06,8912.929789
4,2012-01-05,1.845864e+06,8912.630705
...,...,...,...
5109,2025-12-27,2.303732e+06,8133.478083
5110,2025-12-28,2.303732e+06,8133.478083
5111,2025-12-29,2.303732e+06,8133.478083
5112,2025-12-30,2.303732e+06,8133.478083


In [ ]:
reset_clean_ibge_rural_pop_data_df_2016_2024.to_csv("../../data_files/data/ibge_manaus_fixed_rural_population_data_2016_2024.csv", index=False)